## Heston Model to estimate option prices.

Goal: Use Heston's Stochastic Volatility Model under Real-World Probability Measure.

Data Source: We want to get EOD Historical Data for call options (focus specifically within the S&P 500 for now...?)

Specifications for using the Heston Model:
- Although Black-Scholes is more commonly implemented in the financial markets, its biggest constraint lies in its assumption of constant volatility.
- The Heston model attempts to mitigate this problem by assuming volatility is arbitrary, rather than constant. (Refer to Stochastic Volatility Models)

Other key characteristics of the Heston Model:
- able to factor a correlation between a stock's price and its volatility
- convey volatility as reverting to the mean
- presents a closed-form solution.



Helpful Links:
- [Basic Definition: Investopedia](https://www.investopedia.com/terms/h/heston-model.asp)
- [Heston’s Stochastic Volatility Model Implementation, Calibration and Some Extensions](https://www.maths.univ-evry.fr/pages_perso/crepey/Finance/051111_mikh%20heston.pdf) (We will refer to the materials here for some of our projections.)


Some other food for thought:
- compare the return & pricings over a historical period, see if it aligns well with the actual pricings
- compare with Black-Scholes as a metric to see diff between constant and stochastic volatility

Main Asset Pricing Process (under risk-neutral measure):

Issues with standard option pricings:
- the projections are difficult to predict, as the model has no simple closed-form solutions.

We can therefore look at the characteristic function to help create a closed-form expression to simplify the model.

Some functions to look out for:



In [25]:
#Import Dependencies

import numpy as np
import pandas as pandas

from scipy.integrate import quad
#Need to also find a method to import data to test the model.

In [ ]:
#Define params for Heston Model
class HestonParams:
    def __init__(self, S0, K, r, T, v0, theta, kappa, sigma, rho):
        self.S0 = S0
        self.K = K
        self.r = r
        self.T = T
        self.v0 = v0
        self.theta = theta
        self.kappa = kappa
        self.sigma = sigma
        self.rho = rho

    def lns0(self):
        return np.log(self.S0)
    
    def log_strike(self):
        return np.log(self.K)

In [ ]:
"""
Function for Heston Characteristic Function
"""
def heston_char_func(u, params):
    #To return phi(u), which will be used to create the expectation
    #phi(u) = E[e^{iuln(S_T)}] from 0 to infinite

    #Extract Params
    S0 = params.ln(S0)
    K = params.log_strike
    r = params.r
    T = params.T
    v0 = params.v0
    theta = params.theta
    kappa = params.kappa
    sigma = params.sigma
    rho = params.rho

    # Variable for imaginary number
    i = 1j

    # Define d and g
    a = kappa - rho * sigma * i * u
    b = sigma**2
    c = (i*u + u**2)

    # d(u) = sqrt( (rho*sigma*i*u - kappa)^2 + sigma*(i*u + u^2))
    d = np.sqrt( (rho * sigma * i * u - kappa)**2 + sigma**2 * (i*u + u**2))

    # g(u) = sqrt( (kappa - rho*sigma*i*u - kappa)^2 + sigma^2*(i*u + u^2) )
    num = kappa - rho * sigma * i * u - d
    den = kappa - rho * sigma * i * u + d
    g = num / den

    #Function to calculate B(u, tau)
    exp_neg_dT = np.exp(-d * T)
    B = (kappa - rho * sigma * i * u - d) / b * (1 - exp_neg_dT) / (1 - g * exp_neg_dT)

    #Function to calculate A(u, tau)
    A = (kappa * theta) / b * ((kappa - rho * sigma * i * u - d) * T - 2 * np.log((1 - g * exp_neg_dT) / (1-g)))

    #Function to calculate phi(u) = exp(A + B*v0 + i*u(ln S0 + r*tau))
    exponent = A + B * v0 + i * u(S0 + r * T)

    return np.exp(exponent)

In [28]:
"""
Gil-Palaez Integrands for quad
"""
def integrand_pi2(u, params):
    # Π₂: Re[ e^{-i u k} φ(u) / (i u) ]
    K = params.log_strike()
    phi_u = heston_char_func(u, params)

    i = 1j
    factor = np.exp(-i * u * K) * phi_u / (i * u)

    #We want the factor to be within a real plane, so handle any unknowns
    if np.isnan(factor) or np.isinf(factor):
        return 0.0
    
    return np.real(factor)

def integrand_pi1(u, params):
    # Π₁: Re[ e^{-i u k} φ(u - i) / (i u φ(-i)) ]
    K = params.log_strike()
    
    #Find phi(u - i) first
    u_minus_i = u - 1j
    phi_u_minus_i = heston_char_func(u_minus_i, params)
    phi_neg_i = heston_char_func(-1j, params)

    factor = np.exp(-1j * u * K) * phi_u_minus_i / (-1j * u * phi_neg_i)
    
    return np.real(factor)

In [29]:
"""
Method to handle any singularities for the quad
"""
def safe_integrand_pi2(u, params, eps=1e-10):
    if u < eps:
        return 0.0
    
    return integrand_pi2(u, params)

In [30]:
"""
MAIN OPTIONS PRICING MODEL
"""
def heston_call_prices(params):
    #Compute European calls

    K = params.K()

    #Compute Π₂ = 1/2 + (1/π) ∫₀^∞ Re[ ... ] du
    result_pi2, error_pi2 = quad(integrand_pi2, 0, np.inf, args=(params,), limit=200, epsabs=1e-8)
    pi2 = 0.5 + (1/np.pi) * result_pi2

    #Compute Π₁ = 1/2 + (1/π) ∫₀^∞ Re[ ... ] du
    result_pi1, error_pi1 = quad(integrand_pi1, 0, np.inf, args=(params,), limit=200, epsabs=1e-8)
    pi1 = 0.5 + (1/np.pi) * result_pi1

    #Finally, we can calculate the call price
    call_price = params.S0 * pi1 - params.K * np.exp(-params.r * params.T) * pi2

    return call_price, pi1, pi2, (error_pi1, error_pi2)

In [31]:
"""
Test case: run with only 1 price
"""

def test_heston():
    """
    Initial test params
    """
    params = HestonParams(
        S0=100, K=100, r=0.05, T=1.0,
        v0=0.04, theta=0.04, kappa=2.0,
        sigma=0.0, rho=0.0
    )

    #Define Heston Price:
    heston_price= heston_call_prices(params)

    print(f"Heston price: {heston_price:.6f}")

NameError: name 'self' is not defined